# 🤖 미니 Agent 만들기 - 50줄로 끝내는 Agent의 본질

**목표**: LLM이 도구를 호출하고 → 결과를 보고 → 다음 행동을 결정하는 **Agent의 핵심 루프**를 직접 구현해봅니다.

**왜 프레임워크 없이?** LangChain 같은 프레임워크부터 시작하면 Agent가 영영 블랙박스로 남습니다. 본질은 단순한 `while` 루프 하나임을 직접 눈으로 확인하는 게 목표입니다.

**필요한 것**: [console.anthropic.com](https://console.anthropic.com)에서 API 키 발급 (소액 크레딧으로 충분)

---
## Step 0. 환경 설정

Anthropic SDK 설치 및 API 키 등록.

In [1]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 11.4 MB/s eta 0:00:00


### API 키 등록

**권장**: Colab 좌측 🔑 아이콘 → `ANTHROPIC_API_KEY` 이름으로 추가 → 노트북 액세스 허용

Secrets에 없으면 직접 입력하라는 프롬프트가 뜹니다.

In [2]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ Colab Secrets에서 API 키 로드 완료")
except Exception:
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key를 입력하세요: ")
    print("✅ API 키 입력 완료")

✅ Colab Secrets에서 API 키 로드 완료


---
## Step 1. 도구 정의 - 그냥 평범한 파이썬 함수

Agent가 사용할 도구는 **그냥 함수**입니다. 마법 없음.

여기서는 두 개만:
- `get_time(city)`: 도시별 현재 시각
- `calculator(expression)`: 수식 계산

> ⚠️ `eval()`은 **교육용**입니다. 실서비스에서는 절대 금지 (sandbox 필요).

In [26]:
from datetime import datetime
import zoneinfo

def get_time(city: str) -> str:
    """특정 도시의 현재 시각을 HH:MM 형식으로 반환"""
    tz_map = {
        "Seoul": "Asia/Seoul",
        "Tokyo": "Asia/Tokyo",
        "NewYork": "America/New_York",
        "London": "Europe/London",
    }
    now = datetime.now(zoneinfo.ZoneInfo(tz_map[city]))
    return now.strftime("%H:%M")

def calculator(expression: str) -> str:
    """파이썬 수식 계산 (교육용)"""
    return str(eval(expression)) # ex. 5*10+40

# 도구 레지스트리: 이름 → 함수
TOOLS = {
    "get_time": get_time,
    "calculator": calculator,
}

# 동작 확인
print("서울:", get_time("Seoul"))
print("뉴욕:", get_time("NewYork"))
print("계산:", calculator("365 * 24"))

서울: 03:19
뉴욕: 14:19
계산: 8760


---
## Step 2. LLM에게 도구 명세 알려주기

LLM은 우리 함수를 직접 못 봅니다. **JSON Schema 형태로 명세서**를 줘야 함.

이 명세를 보고 LLM이 **어떤 도구를 호출할지 / 어떤 인자를 줄지** 결정합니다.

In [27]:
TOOLS_SCHEMA = [
    {
        "name": "get_time",
        "description": "특정 도시의 현재 시각(HH:MM)을 반환",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "enum": ["Seoul", "Tokyo", "NewYork", "London"],
                }
            },
            "required": ["city"],
        },
    },
    {
        "name": "calculator",
        "description": "파이썬 수식을 계산",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"],
        },
    },
]

---
## Step 3. Agent 루프 - 이게 전부입니다

**Agent의 본질**:
```
while True:
    LLM에게 물어봄
    if 도구 호출 요청:
        모든 도구 실행 → 결과들을 LLM에게 다시 줌
    else:
        끝, 답변 반환
```

**주의**: Claude는 한 번에 **여러 도구를 병렬로 호출**할 수 있습니다 (예: `get_time("Seoul")`과 `get_time("NewYork")`을 동시에). 그래서 응답에 든 **모든** `tool_use` 블록을 빠짐없이 처리하고, 각각에 대응하는 `tool_result`를 **하나의 user 메시지에 묶어서** 보내야 합니다. 하나라도 빠뜨리면 다음 호출에서 `tool_use ids without tool_result` 에러가 납니다.

추가로 두 가지 안전장치:
- `stop_reason`을 명시적으로 분기 (`end_turn` / `tool_use` / 그 외)
- 도구 실행 중 예외가 나면 `is_error=True`로 LLM에게 알려주기 → LLM이 알아서 복구 시도

In [8]:
─────────────────────────────────────────────
이터레이션 1: client.messages.create() ① 호출
  보낸 messages: [
    {"role": "user", "content": "서울과 뉴욕 시차를 분 단위로"}
  ]
  받은 응답: tool_use → get_time("Seoul"), get_time("NewYork")
            (병렬 호출 가능)
  stop_reason: "tool_use"

  → 우리 코드가 도구 실행: 14:32, 01:32
  → messages에 assistant 응답 + tool_results 추가

─────────────────────────────────────────────
이터레이션 2: client.messages.create() ② 호출
  보낸 messages: [
    user(질문),
    assistant(tool_use 블록들),
    user(tool_results)        ← 새로 추가됨
  ]
  받은 응답: tool_use → calculator("(14*60+32) - (1*60+32)")
  stop_reason: "tool_use"

  → 우리 코드가 계산: 780
  → messages에 또 assistant + tool_result 추가

─────────────────────────────────────────────
이터레이션 3: client.messages.create() ③ 호출
  보낸 messages: [
    user(질문),
    assistant(tool_use 1차),
    user(tool_results 1차),
    assistant(tool_use 2차),  ← 새로 추가됨
    user(tool_result 2차)      ← 새로 추가됨
  ]
  받은 응답: text → "서울이 뉴욕보다 780분 빠릅니다"
  stop_reason: "end_turn"

  → break, 함수 종료
─────────────────────────────────────────────

SyntaxError: invalid character '─' (U+2500) (4276847677.py, line 1)

In [22]:
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"

def run_agent(user_query: str, max_iterations: int = 10, tools_schema:list[str]=[]):
    """LLM ↔ 도구 사이를 오가는 Agent 루프"""
    messages = [{"role": "user", "content": user_query}]
    print(f"👤 사용자: {user_query}\n")

    for i in range(max_iterations):
        # Agent를 호출하고, 응답을 받는 부분
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools_schema,
            messages=messages,
        )

        # 🔧 Agent가 Client가 부여한 도구를 써야한다고 판단
        if response.stop_reason == 'tool_use':
            #  Agent가 생각한 필요한 도구 호출 코드 리스트 (한번에 여러 도구 호출 가능)
            tool_uses = [b for b in response.content if b.type == "tool_use"]
            tool_results = []
            for tu in tool_uses:
                # Agent가 생각하는 필요한 Tool 이름과 입력값
                print(f"🔧 [{i+1}회차] 호출: {tu.name}({tu.input})")
                try:
                    result = str(TOOLS[tu.name](**tu.input)) # 실제 호출 부분
                    is_error = False
                except Exception as e:
                    result = f"Error: {e}"
                    is_error = True
                print(f"📦 결과: {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tu.id,
                    "content": result,
                    "is_error": is_error,
                })
        # ✅ LLM이 "도구 더 안 써도 됨"이라고 판단 → 종료
        elif response.stop_reason == "end_turn":
            final = "(텍스트 응답 없음)"
            for b in response.content:
                if b.type == "text":
                    final = b.text
                    break
            print(f"\n✅ 최종 답변: {final}")
            return final
        # 🛑 예상 외 종료 (max_tokens 등)
        else:
            print(f"⚠️ 예기치 못한 종료: {response.stop_reason}")
            return None

        # 대화 기록 업데이트: assistant 메시지(전체) + 모든 tool_result 한 번에
        #----------------------------
        messages.append({"role": "assistant", "content": response.content}) # get_time({'city':'seoul'})
        messages.append({"role": "user", "content": tool_results}) # 14:00
        #----------------------------
        # 아래와 같이 'role':'tool' 이 되는게 더 자연스럽지만, Anthropic에서는 bot 외에는 모두 User Side contents로 처리
        # messages.append({"role": "tool", "content": tool_results})
        # [원래 흐름]
        # user → assistant → tool → assistant → tool → ... → user

    print("⚠️ 최대 반복 횟수 도달 - 무한 루프 방지 안전장치")

---
## 🚀 실행해보기

**관찰 포인트**:
- LLM이 한 번에 답하는 게 **아니라** 여러 번 도구를 호출하는 흐름이 보입니다.
- 어떤 도구를 어떤 순서로 호출할지는 **LLM이 스스로 결정**합니다.

In [23]:
run_agent("자기 소개 해봐", max_iterations=10, tools_schema=TOOLS_SCHEMA)

👤 사용자: 자기 소개 해봐


✅ 최종 답변: 안녕하세요! 저는 AI 어시스턴트입니다. 😊

저는 다음과 같은 기능들을 도와드릴 수 있습니다:

1. **시간 조회** - Seoul(서울), Tokyo(도쿄), NewYork(뉴욕), London(런던)의 현재 시각을 알려드립니다.

2. **계산** - 수학 계산식을 평가하여 결과를 제공합니다. (덧셈, 뺄셈, 곱셈, 나눗셈 등)

**사용 예시:**
- "서울 현재 시간이 뭐야?"
- "뉴욕의 시간을 알려줄래?"
- "100 + 50 * 2는 몇이야?"
- "3.14 * 5의 제곱근은?"

궁금한 점이나 도움이 필요하신 부분이 있으시면 언제든지 물어봐 주세요! 어떻게 도와드릴까요?


'안녕하세요! 저는 AI 어시스턴트입니다. 😊\n\n저는 다음과 같은 기능들을 도와드릴 수 있습니다:\n\n1. **시간 조회** - Seoul(서울), Tokyo(도쿄), NewYork(뉴욕), London(런던)의 현재 시각을 알려드립니다.\n\n2. **계산** - 수학 계산식을 평가하여 결과를 제공합니다. (덧셈, 뺄셈, 곱셈, 나눗셈 등)\n\n**사용 예시:**\n- "서울 현재 시간이 뭐야?"\n- "뉴욕의 시간을 알려줄래?"\n- "100 + 50 * 2는 몇이야?"\n- "3.14 * 5의 제곱근은?"\n\n궁금한 점이나 도움이 필요하신 부분이 있으시면 언제든지 물어봐 주세요! 어떻게 도와드릴까요?'

In [28]:
run_agent("서울과 뉴욕의 현재 시각 차이를 분 단위로 알려줘",max_iterations=10, tools_schema=TOOLS_SCHEMA)

👤 사용자: 서울과 뉴욕의 현재 시각 차이를 분 단위로 알려줘

🔧 [1회차] 호출: get_time({'city': 'Seoul'})
📦 결과: 03:20
🔧 [1회차] 호출: get_time({'city': 'NewYork'})
📦 결과: 14:20
🔧 [2회차] 호출: calculator({'expression': '(3*60 + 20) - (14*60 + 20)'})
📦 결과: -660

✅ 최종 답변: **서울과 뉴욕의 현재 시각 차이:**

- **서울**: 03:20 (새벽 3시 20분)
- **뉴욕**: 14:20 (오후 2시 20분)

**시간 차이: 660분 (11시간)**

뉴욕이 서울보다 **11시간 뒤**에 있습니다. 즉, 뉴욕은 서울보다 11시간 이전의 시간을 보여주고 있습니다.


'**서울과 뉴욕의 현재 시각 차이:**\n\n- **서울**: 03:20 (새벽 3시 20분)\n- **뉴욕**: 14:20 (오후 2시 20분)\n\n**시간 차이: 660분 (11시간)**\n\n뉴욕이 서울보다 **11시간 뒤**에 있습니다. 즉, 뉴욕은 서울보다 11시간 이전의 시간을 보여주고 있습니다.'

### 좀 더 복잡한 질문

In [29]:
run_agent("지금 도쿄와 런던 시각을 알려주고, 둘의 시차를 시간 단위로 계산해줘", max_iterations=10, tools_schema=TOOLS_SCHEMA)

👤 사용자: 지금 도쿄와 런던 시각을 알려주고, 둘의 시차를 시간 단위로 계산해줘

🔧 [1회차] 호출: get_time({'city': 'Tokyo'})
📦 결과: 03:20
🔧 [1회차] 호출: get_time({'city': 'London'})
📦 결과: 19:20
🔧 [2회차] 호출: calculator({'expression': '3 + 24 - 19'})
📦 결과: 8

✅ 최종 답변: **결과:**

- **도쿄 현재 시각**: 03:20 (오전 3시 20분)
- **런던 현재 시각**: 19:20 (오후 7시 20분)
- **시차**: **8시간** (도쿄가 런던보다 8시간 앞서감)

도쿄는 런던보다 8시간 빠른 시간대에 위치하고 있습니다.


'**결과:**\n\n- **도쿄 현재 시각**: 03:20 (오전 3시 20분)\n- **런던 현재 시각**: 19:20 (오후 7시 20분)\n- **시차**: **8시간** (도쿄가 런던보다 8시간 앞서감)\n\n도쿄는 런던보다 8시간 빠른 시간대에 위치하고 있습니다.'

---
## 🤔 도구 없이 그냥 LLM에게 물어보면?

같은 질문을 **도구 없이** 던져봅시다. LLM은 실시간 시각을 모르므로:
- 답을 거부하거나
- 학습 시점 기준으로 추측 (틀린 답)
- 환각(hallucination)

→ 이게 바로 Agent가 필요한 이유.

In [32]:
run_agent("지금 몇시인지 알려주고, 지금 이 순간 서울과 뉴욕의 시각 차이는 몇 분이야?", max_iterations=10, tools_schema=[])

👤 사용자: 지금 몇시인지 알려주고, 지금 이 순간 서울과 뉴욕의 시각 차이는 몇 분이야?


✅ 최종 답변: 죄송하지만, 저는 현재 시간을 알 수 없습니다. 저는 실시간 정보에 접근할 수 없기 때문입니다.

다만 **일반적인 시간차**는 알려드릴 수 있습니다:

- **서울**: 한국 표준시 (KST, UTC+9)
- **뉴욕**: 동부 표준시 또는 동부 일광절약시간
  - 표준시: EST (UTC-5) → 차이 **14시간**
  - 일광절약시간: EDT (UTC-4) → 차이 **13시간**

따라서:
- **3월~11월**: 서울이 뉴욕보다 **13시간 앞**
- **11월~3월**: 서울이 뉴욕보다 **14시간 앞**

현재 정확한 시간을 알고 싶으시면 검색이나 시계 앱을 확인하시면 됩니다!


'죄송하지만, 저는 현재 시간을 알 수 없습니다. 저는 실시간 정보에 접근할 수 없기 때문입니다.\n\n다만 **일반적인 시간차**는 알려드릴 수 있습니다:\n\n- **서울**: 한국 표준시 (KST, UTC+9)\n- **뉴욕**: 동부 표준시 또는 동부 일광절약시간\n  - 표준시: EST (UTC-5) → 차이 **14시간**\n  - 일광절약시간: EDT (UTC-4) → 차이 **13시간**\n\n따라서:\n- **3월~11월**: 서울이 뉴욕보다 **13시간 앞**\n- **11월~3월**: 서울이 뉴욕보다 **14시간 앞**\n\n현재 정확한 시간을 알고 싶으시면 검색이나 시계 앱을 확인하시면 됩니다!'

---
## ✏️ 직접 시도해보기

새로운 도구를 만들어서 LLM에게 주어주고, 해당 도구를 사용해서 문제를 해결하는 시나리오를 만들어보세요.

In [ ]:
def your_tool_name(param1: str, param2: int = 10) -> str:
    """
    [Claude는 이 docstring을 안 봄. description은 schema에 따로!]
    여기는 다른 개발자(혹은 미래의 자기)를 위한 메모.
    """
    # 1. 입력 검증
    if not param1:
        return "Error: param1 cannot be empty"

    # 2. 본 작업
    try:
        result = do_something(param1, param2)
    except SpecificException as e:
        return f"Error: {e}"  # ← Claude가 보고 복구할 수 있게

    # 3. 결과를 LLM-친화적 형식으로
    if isinstance(result, dict):
        return json.dumps(result, ensure_ascii=False)
    return str(result)


# Schema도 같이 작성하기
your_tool_schema = {
    "name": "your_tool_name",
    "description": (
        "[무엇을 하는지 한 문장]. "
        "[입력 의미와 단위]. "
        "[언제 쓰면 안 되는지]."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "param1": {"type": "string", "description": "..."},
            "param2": {"type": "integer", "description": "...", "default": 10}
        },
        "required": ["param1"]
    }
}

In [33]:
# ❌ 나쁜 예
{
    "name": "calculate",
    "description": "계산을 합니다",   # 너무 모호
    "input_schema": {
        "type": "object",
        "properties": {"x": {"type": "string"}},  # x가 뭔지?
    }
}

# ✅ 좋은 예
{
    "name": "calculate_compound_interest",
    "description": (
        "복리 이자를 계산합니다. "
        "원금, 연이율(%), 기간(년)을 받아 최종 금액을 KRW로 반환합니다. "
        "단리 계산이 필요하면 이 도구 대신 calculator를 사용하세요."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "principal": {
                "type": "number",
                "description": "원금 (KRW). 양수여야 함"
            },
            "rate_percent": {
                "type": "number",
                "description": "연이율 (%). 예: 5%면 5"
            },
            "years": {
                "type": "integer",
                "description": "투자 기간 (년)"
            }
        },
        "required": ["principal", "rate_percent", "years"]
    }
}

{'name': 'calculate_compound_interest',
 'description': '복리 이자를 계산합니다. 원금, 연이율(%), 기간(년)을 받아 최종 금액을 KRW로 반환합니다. 단리 계산이 필요하면 이 도구 대신 calculator를 사용하세요.',
 'input_schema': {'type': 'object',
  'properties': {'principal': {'type': 'number',
    'description': '원금 (KRW). 양수여야 함'},
   'rate_percent': {'type': 'number', 'description': '연이율 (%). 예: 5%면 5'},
   'years': {'type': 'integer', 'description': '투자 기간 (년)'}},
  'required': ['principal', 'rate_percent', 'years']}}

In [ ]:
# 여기에 질문을 바꿔서 실행해보세요
run_agent("", max_iterations=10, tools_schema=your_tool_schema)

---
## 🌱 다음 단계

이 미니 Agent를 확장한다면:

| 추가 요소 | 무엇을 배우게 되는가 |
|---|---|
| 새 도구 (웹 검색, 파일 읽기) | 도구의 다양성 |
| 메모리 (이전 대화 기록 보존) | Stateful Agent |
| 여러 도구 동시 호출 (parallel tool use) | 효율 vs 의존성 |
| 외부 API 도구 (검색/지도/DB) | 실제 응용 |
| `smolagents`, `LangGraph` 등 프레임워크 | 위의 루프가 어떻게 추상화되는지 |

**핵심 메시지**: 어떤 프레임워크를 쓰든, 어떤 도구를 추가하든, **Agent의 본질은 위의 `while` 루프 하나**입니다. 이걸 이해했다면 Agent의 80%를 이해한 것.